# Dev/test split

**Summary**
- 480 queries split into dev 241 / test 239 (seed 2026); odd-sized buckets round towards dev.
- 5 near-duplicate pairs kept together: 3 in dev, 2 in test.
- Difficulty buckets match to within one query; median query length (3 words) and median Exact matches (10) are equal on both sides.
- Dev has slightly more labels per query on average (511 vs 455), driven by a few very large queries; medians match.

In [1]:
import pandas as pd

from product_search.data.splits import BUCKETS, difficulty_bucket, load_split
from product_search.data.wands import load_wands

data = load_wands()
split = load_split()
queries = data.queries.set_index("query_id")
side = pd.Series({q: "dev" for q in split.dev} | {q: "test" for q in split.test}, name="side")

assert not set(split.dev) & set(split.test), "a query is on both sides"
assert len(side) == len(queries), "not every query is assigned"
print(f"dev {len(split.dev)} / test {len(split.test)}, seed {split.seed}")

dev 241 / test 239, seed 2026


## 1. Near-duplicates kept together

In [2]:
for group in split.duplicate_groups:
    print(side[group[0]], [queries.loc[q, "query"] for q in group])

dev ['leather chairs', 'leather chair']
dev ['orren ellis l shape desk', 'orren ellis l shaped desk']
test ['kitchen anti fatigue mats', 'anti fatigue mat']
test ['e12/candelabra', 'ligth bulb']
dev ['bistro table and chairs', 'bistro sets patio']


## 2. Balance

In [3]:
per_query = (
    data.labels.pivot_table(
        index="query_id", columns="label", values="grade", aggfunc="count", fill_value=0
    )
    .reindex(queries.index, fill_value=0)
)
stats = pd.DataFrame(
    {
        "side": side,
        "bucket": per_query["Exact"].map(difficulty_bucket),
        "words": queries["query"].str.split().str.len(),
        "labels": per_query.sum(axis=1),
        "exact": per_query["Exact"],
    }
)
display(pd.crosstab(stats["bucket"], stats["side"]).reindex(BUCKETS))
display(stats.groupby("side")[["words", "labels", "exact"]].agg(["mean", "median"]).round(1))

side,dev,test
bucket,,
0,51,50
1-10,70,70
11-100,83,82
>100,37,37


words        labels        exact       
      mean median   mean median  mean median
side                                        
dev    3.4    3.0  511.3  218.0  56.4   10.0
test   3.4    3.0  454.6  214.0  49.7   10.0